In [1]:
%load_ext autoreload
import numpy as np
import jax, scipy
from jax import numpy as jnp
from qihd.utils.jax_utils import jax_device

from qihd import MIQPSolver, QIHD, PDQP
from qihd.problems.lcqp import LCQP

In [2]:
# Set up device; optional if device = 'cpu'
device = "cpu"
jax.config.update("jax_platforms", jax_device(device))

In [3]:
# Generate a random LCQP instance

seed = 19
np.random.seed(seed)

n = 20
m = 10
k = 3
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
lbs = np.random.uniform(-2, 2, (n))
x = lbs + np.random.uniform(0.1, 1, (n))
ubs = x + np.random.uniform(0.1, 1, (n))
A = np.random.normal(0, 1, (m, n))
b = np.random.normal(0, np.sqrt(n), (m))
infeas = A @ x > b
A[infeas] *= -1
b[infeas] *= -1
C = np.random.uniform(0, 1, (k, n))
d = C @ x
prob = LCQP(Q, w, A=A, b=b, C=C, d=d, bounds=(lbs, ubs))

In [5]:
# Set up backend (QIHD)
n_shots = 100
n_steps = 10000
seed = 42
device = "cpu"
lc_pr = 10
slow_a = False
backend = QIHD(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, lc_pr=lc_pr, slow_a=slow_a)

# Set up refiner; optional for PDQP
# Set up MIQPSolver; solve
model = MIQPSolver(prob, backend) # default refiner = PDQP
res = model.solve()

# Validate results
xs, cnts = res.refined_samples, res.sample_counts

objs = np.array([prob.obj(x) for x in xs])
maxvios = np.array([prob.max_vios(x) for x in xs])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(cnts[feas][succ]) / n_shots
print(f"----- Running QiHD -----\nBackend: {backend.__class__.__name__}; Refiner: PDQP.")
print(f"Incumbent minimum: {minima}, feasibility violations: {minimizer_vios}, success probability : {succ_prob}.")
assert minima <= -455.8266

----- Running QiHD -----
Backend: QIHD; Refiner: PDQP.
Incumbent minimum: -455.8266564262336, feasibility violations: 5.2403068140804976e-05, success probability : 0.01.


In [ ]:
# Test refiner: PDQP
samples = np.random.normal(0, 1, (n_shots, prob.nvar))
pdqp = PDQP(problem=prob, device=device, iterations=10000)
res = pdqp.pdqp_main(samples)
objs = np.array([prob.obj(x) for x in res])
maxvios = np.array([prob.max_vios(x) for x in res])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(succ) / n_shots
print(minima, minimizer_vios, succ_prob)

-455.8299379840335 5.151018818061459e-05 0.01
